# Convert YOLO11n SimAM+CA Augmentation `best.pt` to TFLite

This notebook converts the augmentation experiment checkpoint:

`simam_augmentation_run_simam_ca/weights/best.pt`

The checkpoint is a YOLO segmentation model, not the 4-class Android classifier. Its model labels are `BG` and `WSSV`. The Android app's current `assets/labels.txt` is kept as a separate 4-class classifier label file: `Healthy`, `BG`, `WSSV`, `WSSV_BG`.

Do not overwrite `assets/labels.txt` with the 2-class segmentation labels unless the Android runtime is changed to load this segmentation model with segmentation postprocessing.

## Install export dependencies

Run this once in Colab or a fresh Python environment. A runtime restart may be needed after TensorFlow-related packages are installed.

In [ ]:
%pip -q install "typing_extensions>=4.15.0" "ml_dtypes>=0.5.4" "sympy>=1.13.1" "onnx>=1.12.0,<2.0.0" "onnxslim>=0.1.82" "onnx2tf>=1.26.3,<1.29.0" "onnx-graphsurgeon>=0.3.26" "sng4onnx>=1.0.1" ai-edge-litert onnxruntime pandas numpy

## Configure paths

If this notebook is opened from the project root, the defaults should work. In Colab, set `REPO_ROOT` to the mounted project folder before running the cell.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import sys
import zipfile

# Change this if your working directory is not the repository root.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'yolov11n_attention').exists():
    # Common Colab fallback. Edit this path if your Drive layout is different.
    REPO_ROOT = Path('/content/drive/MyDrive/CVio_Shrimp_Disease_Classification_Capstone_SU26')

AUG_DIR = REPO_ROOT / 'yolov11n_attention' / 'yolov11n_simam_NhomA' / 'augmentation'
RUN_ROOT = AUG_DIR / 'yolo11n_simam_augmentation_run_all_results_20260615_080026'
TARGET_RUN = 'simam_augmentation_run_simam_ca'
ZIP_PATH = AUG_DIR / 'yolo11n_simam_augmentation_run_all_results_20260615_080026.zip'
BEST_PT = RUN_ROOT / TARGET_RUN / 'weights' / 'best.pt'

ANDROID_ASSETS_DIR = REPO_ROOT / 'LiteRT-for-Android' / 'app' / 'src' / 'main' / 'assets'
ANDROID_LABELS_PATH = ANDROID_ASSETS_DIR / 'labels.txt'

# Keep export output and work files on short Windows-friendly paths.
EXPORT_DIR = AUG_DIR / 'tflite_export_simam_ca'
WORK_DIR = REPO_ROOT / 'tflite_work_simam_ca'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT:', REPO_ROOT)
print('BEST_PT:', BEST_PT)
print('ZIP_PATH:', ZIP_PATH)
print('EXPORT_DIR:', EXPORT_DIR)
print('WORK_DIR:', WORK_DIR)
print('ANDROID_LABELS_PATH:', ANDROID_LABELS_PATH)

## Extract `best.pt` if needed

In [ ]:
if not BEST_PT.exists():
    if not ZIP_PATH.exists():
        raise FileNotFoundError(f'Missing both checkpoint and zip artifact: {BEST_PT} / {ZIP_PATH}')
    entry_name = f'{TARGET_RUN}/weights/best.pt'
    BEST_PT.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as archive:
        if entry_name not in archive.namelist():
            raise FileNotFoundError(f'Missing {entry_name} in {ZIP_PATH}')
        with archive.open(entry_name) as src, BEST_PT.open('wb') as dst:
            shutil.copyfileobj(src, dst)
    print('Extracted:', BEST_PT)
else:
    print('Checkpoint already exists:', BEST_PT)

print('Size MB:', round(BEST_PT.stat().st_size / (1024 ** 2), 2))

## Use the local custom Ultralytics source

The checkpoint was trained with custom SimAM/Coordinate-Attention modules. Use the repository's `yolov11n_attention/ultralytics` source instead of a plain PyPI install.

In [ ]:
LOCAL_ULTRALYTICS = REPO_ROOT / 'yolov11n_attention' / 'ultralytics'
if not LOCAL_ULTRALYTICS.exists():
    raise FileNotFoundError(f'Missing local custom Ultralytics source: {LOCAL_ULTRALYTICS}')

os.environ['YOLO_CONFIG_DIR'] = str(REPO_ROOT / 'Ultralytics')
os.environ['TF_USE_LEGACY_KERAS'] = '1'
sys.path.insert(0, str(LOCAL_ULTRALYTICS))

import torch
import ultralytics
import ultralytics.nn.modules.conv as conv_modules
from ultralytics import YOLO

# Force CPU-side export dependency checks. This avoids an unnecessary onnxruntime-gpu install.
torch.cuda.is_available = lambda: False

print('Ultralytics version:', ultralytics.__version__)
print('Ultralytics source:', ultralytics.__file__)

## Patch CoordAtt to match the training notebook

The augmentation training notebook used mean pooling inside `CoordAtt.forward`. The local source currently uses `pool_h/pool_w`, which older saved modules do not contain. This patch restores the training-time forward pass before loading/exporting the checkpoint.

In [ ]:
def coordatt_forward_training_compatible(self, x):
    identity = x
    _, _, h, w = x.size()
    x_h = x.mean(dim=3, keepdim=True)
    x_w = x.mean(dim=2, keepdim=True).permute(0, 1, 3, 2)
    y = torch.cat([x_h, x_w], dim=2)
    y = self.act(self.bn1(self.conv1(y)))
    x_h, x_w = torch.split(y, [h, w], dim=2)
    x_w = x_w.permute(0, 1, 3, 2)
    a_h = self.conv_h(x_h).sigmoid()
    a_w = self.conv_w(x_w).sigmoid()
    if getattr(self, 'proj', None) is not None:
        identity = self.proj(identity)
    return identity * a_h * a_w

conv_modules.CoordAtt.forward = coordatt_forward_training_compatible
print('Patched CoordAtt.forward for augmentation checkpoint export.')

## Load checkpoint and verify labels

In [ ]:
model = YOLO(str(BEST_PT))
model_names = [model.names[i] for i in sorted(model.names)]
print('Task:', model.task)
print('Model names:', model.names)

expected_segmentation_labels = ['BG', 'WSSV']
if model_names != expected_segmentation_labels:
    raise ValueError(f'Unexpected segmentation labels: {model_names}')

asset_labels = []
if ANDROID_LABELS_PATH.exists():
    asset_labels = [line.strip() for line in ANDROID_LABELS_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
print('Android classifier labels.txt:', asset_labels)
print('Use segmentation labels for this model:', model_names)

## Export float32 TFLite

The training `args.yaml` uses `imgsz: 640`, so the export keeps `imgsz=640`. The output is a segmentation TFLite model with detection and mask tensors.

In [ ]:
import numpy as np

IMG_SIZE = 640
WORK_BEST_PT = WORK_DIR / 'best.pt'
shutil.copy2(BEST_PT, WORK_BEST_PT)
(WORK_DIR / 'best_saved_model' / 'variables').mkdir(parents=True, exist_ok=True)

# onnx2tf checks for this file in the current working directory even for float32 export.
previous_cwd = Path.cwd()
os.chdir(WORK_DIR)
calibration_file = WORK_DIR / 'calibration_image_sample_data_20x128x128x3_float32.npy'
created_calibration = False
if not calibration_file.exists():
    np.save(calibration_file, np.random.rand(20, 128, 128, 3).astype(np.float32))
    created_calibration = True

try:
    work_model = YOLO(str(WORK_BEST_PT))
    export_result = work_model.export(format='tflite', imgsz=IMG_SIZE, int8=False, half=False, nms=False)
finally:
    os.chdir(previous_cwd)
    if created_calibration and calibration_file.exists():
        calibration_file.unlink()

print('Ultralytics export result:', export_result)

source_float32 = WORK_DIR / 'best_saved_model' / 'best_float32.tflite'
source_float16 = WORK_DIR / 'best_saved_model' / 'best_float16.tflite'
if not source_float32.exists():
    raise FileNotFoundError(f'Missing expected float32 export: {source_float32}')

output_tflite = EXPORT_DIR / 'yolo11n_simam_ca_augmentation_best_float32.tflite'
output_tflite_float16 = EXPORT_DIR / 'yolo11n_simam_ca_augmentation_best_float16.tflite'
shutil.copy2(source_float32, output_tflite)
if source_float16.exists():
    shutil.copy2(source_float16, output_tflite_float16)

print('Copied float32 TFLite:', output_tflite, round(output_tflite.stat().st_size / (1024 ** 2), 2), 'MB')
if output_tflite_float16.exists():
    print('Copied float16 TFLite:', output_tflite_float16, round(output_tflite_float16.stat().st_size / (1024 ** 2), 2), 'MB')

## Write segmentation labels and metadata

In [ ]:
seg_labels_path = EXPORT_DIR / 'yolo11n_simam_ca_seg_labels.txt'
seg_labels_path.write_text('\n'.join(model_names) + '\n', encoding='utf-8')

metadata = {
    'source_checkpoint': str(BEST_PT),
    'source_zip': str(ZIP_PATH),
    'work_checkpoint': str(WORK_BEST_PT),
    'task': model.task,
    'imgsz': IMG_SIZE,
    'float32_tflite_path': str(output_tflite),
    'float16_tflite_path': str(output_tflite_float16) if output_tflite_float16.exists() else '',
    'segmentation_labels_path': str(seg_labels_path),
    'segmentation_labels': model_names,
    'android_classifier_labels_txt': asset_labels,
    'input': {'name': 'images', 'shape': [1, IMG_SIZE, IMG_SIZE, 3], 'dtype': 'float32'},
    'outputs': [
        {'name': 'Identity', 'shape': [1, 38, 8400], 'dtype': 'float32'},
        {'name': 'Identity_1', 'shape': [1, 160, 160, 32], 'dtype': 'float32'},
    ],
    'note': 'This is a 2-class YOLO segmentation model. Do not use the 4-class classifier labels.txt for segmentation postprocessing.',
}
metadata_path = EXPORT_DIR / 'yolo11n_simam_ca_augmentation_best_tflite_metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

print('Wrote:', seg_labels_path)
print('Wrote:', metadata_path)
print(json.dumps(metadata, indent=2))

## Smoke test the exported TFLite tensors

In [ ]:
try:
    from ai_edge_litert.interpreter import Interpreter
    backend = 'ai_edge_litert'
except Exception:
    import tensorflow as tf
    Interpreter = tf.lite.Interpreter
    backend = 'tensorflow.lite'

for tflite_path in [output_tflite, output_tflite_float16]:
    if not tflite_path.exists():
        continue
    interpreter = Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    print('Model:', tflite_path.name)
    print('Interpreter backend:', backend)
    print('Inputs:')
    for item in input_details:
        print({k: item[k] for k in ['name', 'shape', 'dtype', 'index']})
    print('Outputs:')
    for item in output_details:
        print({k: item[k] for k in ['name', 'shape', 'dtype', 'index']})

## Optional copy to Android assets

Keep this disabled unless the Android app has a segmentation runtime/postprocessor. The current `ShrimpClassifier` expects a flat 4-class classifier output and will reject or misread YOLO segmentation tensors.

In [ ]:
COPY_TO_ANDROID_ASSETS = False

if COPY_TO_ANDROID_ASSETS:
    ANDROID_ASSETS_DIR.mkdir(parents=True, exist_ok=True)
    android_tflite = ANDROID_ASSETS_DIR / output_tflite.name
    android_seg_labels = ANDROID_ASSETS_DIR / seg_labels_path.name
    shutil.copy2(output_tflite, android_tflite)
    shutil.copy2(seg_labels_path, android_seg_labels)
    print('Copied model:', android_tflite)
    print('Copied segmentation labels:', android_seg_labels)
else:
    print('Skipped Android asset copy. Enable only after app segmentation support is implemented.')